# backprop-pop-outgrad-loop — faded example 1: Complete the parent-accumulation step in the reverse-pass driver

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `backprop-pop-outgrad-loop`. Running the beacon reports progress on the `Backprop: backprop pop-outgrad loop` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: backprop pop-outgrad loop` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backprop-pop-outgrad-loop`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backprop-pop-outgrad-loop"
DD_SUBTOPIC = "Backprop: backprop pop-outgrad loop"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In the non-leaf branch of the backprop loop, each parent's `back_fn` produces a gradient contribution that must be **added** into that parent's slot in the `grads` dict, never overwritten. Using `grads.get(pid, 0) + gp` is what lets a parent that appears on multiple paths (a diamond DAG) collect all its incoming gradient before it is popped.

## Faded exercise 1

### Faded — finish the parent-accumulation line

The `backprop` driver below is complete except for the line that writes each parent's gradient contribution back into the `grads` accumulator. Complete it so that contributions **accumulate** (do not overwrite). The graph is `out = z*z` (a diamond on leaf `z`), where correct behaviour gives `z.grad == 2*z`.

**Fill in:** Accumulate the parent's gradient contribution `gp` into `grads[pid]`, adding to any value already there (defaulting to 0).

In [ ]:
import numpy as np
import torch as t
from einops import rearrange, reduce, repeat
np.random.seed(0); t.manual_seed(0)

class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args; self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array; self.recipe = recipe; self.grad = None

def _mul_back0(go, out, x, y):  return go * y
def _mul_back1(go, out, x, y):  return go * x

def backprop(end_node, end_grad, sorted_graph, back_funcs) -> None:
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)
        if node.recipe is None:
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            bf = back_funcs[(node.recipe.func, argnum)]
            gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp

t.manual_seed(0)
z = MiniTensor(t.randn(4))
out_arr = z.array * z.array
out = MiniTensor(out_arr, Recipe('mul', (z.array, z.array), {}, {0: z, 1: z}))
back_funcs = {('mul', 0): _mul_back0, ('mul', 1): _mul_back1}
backprop(out, t.ones_like(out.array), [out, z], back_funcs)


def _test():
    assert z.grad is not None, 'z.grad was never set'
    expected = 2 * z.array
    assert t.allclose(z.grad, expected), f'expected 2z, got {z.grad}'
    # both diamond paths must have accumulated; overwrite would give z (not 2z)
    assert not t.allclose(z.grad, z.array), 'looks like one path was overwritten'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
import torch as t
from einops import rearrange, reduce, repeat
np.random.seed(0); t.manual_seed(0)

class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args; self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array; self.recipe = recipe; self.grad = None

def _mul_back0(go, out, x, y):  return go * y
def _mul_back1(go, out, x, y):  return go * x

def backprop(end_node, end_grad, sorted_graph, back_funcs) -> None:
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)
        if node.recipe is None:
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            bf = back_funcs[(node.recipe.func, argnum)]
            gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp

t.manual_seed(0)
z = MiniTensor(t.randn(4))
out_arr = z.array * z.array
out = MiniTensor(out_arr, Recipe('mul', (z.array, z.array), {}, {0: z, 1: z}))
back_funcs = {('mul', 0): _mul_back0, ('mul', 1): _mul_back1}
backprop(out, t.ones_like(out.array), [out, z], back_funcs)
```
</details>